# Fine Tuning Mistral to improve CMR report generation (DPO)



In [ ]:
# Clone the repo (skip if already cloned)
!git clone https://github.com/JeremieTarantop/Cardiac-Diagnostic-CMR-report-.git repo_cmr
%cd repo_cmr

Cloning into 'repo_cmr'...
remote: Enumerating objects: 14279, done.
remote: Counting objects: 100% (14279/14279), done.
remote: Compressing objects: 100% (13665/13665), done.
remote: Total 14279 (delta 125), reused 14237 (delta 99), pack-reused 0 (from 0)
Receiving objects: 100% (14279/14279), 31.80 MiB | 30.64 MiB/s, done.
Resolving deltas: 100% (125/125), done.
/content/repo_cmr


In [ ]:
# Install dependencies and enable GPU
import os
os.environ["USE_TRANSFORMERS"] = "1"
os.environ["USE_CUDA"] = "1"  # use GPU on Colab

!pip install -q transformers torch pandas numpy scipy

In [ ]:
# Install, the quotes matter
!pip -q install -U "bitsandbytes>=0.46.1" trl datasets accelerate peft transformers

# Verify bitsandbytes is actually importable and the version is correct
import bitsandbytes as bnb, importlib
print("bitsandbytes version:", bnb.__version__)
print("bitsandbytes path:", bnb.__file__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.6 MB/s eta 0:00:00
bitsandbytes version: 0.49.2
bitsandbytes path: /usr/local/lib/python3.12/dist-packages/bitsandbytes/__init__.py


In [ ]:
!pip -q install -U "trl>=0.9.6" "transformers>=4.41.0" "accelerate>=0.31.0" datasets peft bitsandbytes
import trl, transformers, accelerate
print("trl:", trl.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)


trl: 0.28.0
transformers: 5.2.0
accelerate: 1.12.0


## Using Mistral 7b to generate reports based on only the small ECG report


## Fine tuning Mistral using DPO framework with 1 file

In [ ]:
# CELL 5, replace your whole file discovery block with this

from pathlib import Path

ROOT = Path(".")
DATA_DIR = ROOT / "synthetic_CMR_llama1B"   # folder in your repo

gt_path   = DATA_DIR / "GPT_5.2_cmr_report_001.txt"
gen1_path = DATA_DIR / "llama1_cmr_report_0001.txt"
gen2_path = DATA_DIR / "llama1_cmr_report_0001_v2.txt"

assert gt_path.exists(),   f"Missing: {gt_path}"
assert gen1_path.exists(), f"Missing: {gen1_path}"
assert gen2_path.exists(), f"Missing: {gen2_path}"

PROMPT_PATH = ROOT / "prompt_v1.txt"
assert PROMPT_PATH.exists(), f"Missing prompt file: {PROMPT_PATH.resolve()}"
base_prompt = PROMPT_PATH.read_text(encoding="utf-8").strip()


## Using Mistral to judge the quality of the generated reports

In [ ]:
# This loads Mistral as a judge and picks chosen vs rejected among the two generated reports.

import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

JUDGE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
if judge_tokenizer.pad_token is None:
    judge_tokenizer.pad_token = judge_tokenizer.eos_token

judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    )

def _extract_winner(text: str) -> str:
    """
    Expect the judge to output a single letter: A or B.
    """
    m = re.search(r"\b([AB])\b", text.strip(), flags=re.IGNORECASE)
    if not m:
        return "A"
    return m.group(1).upper()

def judge_pick(report_a: str, report_b: str, gt_text: str | None = None) -> str:
    """
    Returns "A" or "B".
    If gt_text is provided, it is used as style and coverage reference,
    but the judge is instructed to penalize unsupported claims.
    """
    gt_block = ""
    if gt_text is not None and gt_text.strip():
        gt_block = f"\n\nGROUND TRUTH (reference style and coverage, do not copy):\n{gt_text}\n"

    judge_prompt = f"""
You are a strict clinical evaluator. Choose which report is better for the same ECG input.
Priorities:
1) Faithfulness and safety: do not invent findings, numbers, diagnoses, or CMR details not supported by the input.
2) Clinical correctness: prefer accurate interpretation and appropriate uncertainty.
3) Clarity and structure.

Output exactly one character: A or B.

REPORT A:
{report_a}

REPORT B:
{report_b}
{gt_block}
""".strip()

    inputs = judge_tokenizer(judge_prompt, return_tensors="pt").to(judge_model.device)
    with torch.no_grad():
        out = judge_model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            temperature=0.0,
            pad_token_id=judge_tokenizer.eos_token_id,
            eos_token_id=judge_tokenizer.eos_token_id,
        )
    decoded = judge_tokenizer.decode(out[0], skip_special_tokens=True)
    completion = decoded[len(judge_prompt):].strip() if decoded.startswith(judge_prompt) else decoded.strip()
    return _extract_winner(completion)


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:

from pathlib import Path

gt_text   = gt_path.read_text(encoding="utf-8").strip()
gen1_text = gen1_path.read_text(encoding="utf-8").strip()
gen2_text = gen2_path.read_text(encoding="utf-8").strip()

winner = judge_pick(gen1_text, gen2_text, gt_text=gt_text)

if winner == "A":
    chosen_text, rejected_text = gen1_text, gen2_text
    chosen_path, rejected_path = gen1_path, gen2_path
else:
    chosen_text, rejected_text = gen2_text, gen1_text
    chosen_path, rejected_path = gen2_path, gen1_path

print("Ground truth file:", gt_path)
print("Gen A file:", gen1_path)
print("Gen B file:", gen2_path)
print("\nJudge winner:", winner)
print("Chosen file:", chosen_path)
print("Rejected file:", rejected_path)

x_prompt = base_prompt

example = {
    "prompt": x_prompt,
    "chosen": chosen_text,
    "rejected": rejected_text,
}

example["chosen"]


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Ground truth file: synthetic_CMR_llama1B/GPT_5.2_cmr_report_001.txt
Gen A file: synthetic_CMR_llama1B/llama1_cmr_report_0001.txt
Gen B file: synthetic_CMR_llama1B/llama1_cmr_report_0001_v2.txt

Judge winner: A
Chosen file: synthetic_CMR_llama1B/llama1_cmr_report_0001.txt
Rejected file: synthetic_CMR_llama1B/llama1_cmr_report_0001_v2.txt


"Clinical Indication: This patient has been diagnosed with hypertrophic obstructive cardiomyopathy (HOCM) based on his history of angina pectoris, dyslipidemia, and elevated troponin levels. He is currently being treated with beta blocker therapy for hypertension and statins for hypertriglyceridemia. The patient's ECG shows evidence of left ventricular hypertrophy (LVH) with normal LVEF but increased RBBB and ST segment depression. His CMR report confirms this finding by showing enlarged myocardium with reduced perfusion in the right atrium and right ventricle. There is also evidence of pericardial effusion (PCE) with decreased PVC amplitude in leads V1-V6. These findings support the clinical indications and further confirm HOCM as the underlying etiology.\n\nFindings (brief):\n- Left Ventricular Hypertrophy (LVH) with Normal LVEF\n- Right Atrial Enlargement (RAE)\n- Reduced Perfusion in"


Looking at a preview of the ground truth and generated reports

#### Set up TRL DPOTrainer with a frozen reference model

Downloading MedGemma as a potential reference model

In [ ]:
# from huggingface_hub import hf_hub_download

# token = "hf_XXX"
# hf_hub_download(
#     repo_id="google/medgemma-4b-it",
#     filename="config.json",
#     token=token
# )
# print("Access confirmed")


In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer


# 1) Dataset
train_ds = Dataset.from_list([example])

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# cfg = AutoConfig.from_pretrained(MODEL_ID)
# cfg.tie_word_embeddings = False

# 2) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3) Load policy model on GPU (trainable via LoRA)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
)

# 4) Load reference model on CPU (frozen anchor)
model_ref = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": "cpu"},
    torch_dtype=torch.float16,
)

model_ref.eval()
for p in model_ref.parameters():
    p.requires_grad_(False)

# 5) LoRA on policy model only
peft_cfg = LoraConfig(
    r=16,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# 6) DPO config
dpo_args = DPOConfig(
    output_dir="dpo_cmr_sanity",
    beta=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=1e-5,
    logging_steps=1,
    save_steps=50,
    fp16=True,
    gradient_checkpointing=True,
)

# 7) Trainer
trainer_kwargs = dict(
    model=model,
    ref_model=model_ref,
    args=dpo_args,
    train_dataset=train_ds,
    peft_config=peft_cfg,
)

try:
    trainer = DPOTrainer(**trainer_kwargs, processing_class=tokenizer)
except TypeError:
    trainer = DPOTrainer(**trainer_kwargs, tokenizer=tokenizer)

print("Ready. Next: trainer.train()")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Extracting prompt in train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Ready. Next: trainer.train()



## Visualization of "Before" and "After" finetuning

In [ ]:
import torch

def generate_report(model, tokenizer, prompt: str, max_new_tokens: int = 400, temperature: float = 0.7, top_p: float = 0.95):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded.strip()

def show_reports(prompt, baseline, gen1, gen2, chosen, rejected, ground_truth, new_report=None,
                 gen1_name="GEN 1", gen2_name="GEN 2", chosen_name="CHOSEN", rejected_name="REJECTED"):
    print("=" * 100)
    print("PROMPT\n")
    print(prompt)

    print("\n" + "=" * 100)
    print("BASELINE (Mistral before DPO)\n")
    print(baseline)


    print("\n" + "=" * 100)
    print(f"{chosen_name}\n")
    print(chosen)

    print("\n" + "=" * 100)
    print(f"{rejected_name}\n")
    print(rejected)

    print("\n" + "=" * 100)
    print("GROUND TRUTH\n")
    print(ground_truth)

    if new_report is not None:
        print("\n" + "=" * 100)
        print("NEW REPORT (after DPO)\n")
        print(new_report)

    print("\n" + "=" * 100)

# Baseline generation from the pre DPO model (your `model`, not trainer.model)
baseline_text = generate_report(model, tokenizer, x_prompt, max_new_tokens=450, temperature=0.7, top_p=0.95)

# Show everything before training
show_reports(
    prompt=x_prompt,
    baseline=baseline_text,
    gen1=gen1_text,
    gen2=gen2_text,
    chosen=chosen_text,
    rejected=rejected_text,
    ground_truth=gt_text,
    gen1_name=f"GEN 1 ({gen1_path.name})",
    gen2_name=f"GEN 2 ({gen2_path.name})",
    chosen_name=f"CHOSEN (Judge picked {winner}, {chosen_path.name})",
    rejected_name=f"REJECTED ({rejected_path.name})",
)

# Optional: print trainable params (LoRA)
trainer.model = trainer._wrap_model(trainer.model, training=True)
trainer.model.print_trainable_parameters()

def count_params(m):
    total = 0
    trainable = 0
    for p in m.parameters():
        n = p.numel()
        total += n
        if p.requires_grad:
            trainable += n
    return total, trainable

total_params, trainable_params = count_params(trainer.model)
ratio = trainable_params / total_params
print(f"\nTotal parameters:      {total_params:,}")
print(f"Trainable parameters:  {trainable_params:,}")
print(f"Trainable ratio:       {ratio:.6%}")

PROMPT

A.1 Diagnosis Guider Prompt 
# Your task:  Interpret the provided ECG data, identify key features and abnormalities in each lead, and generate a clinical diagnosis that is supported by the observed evidence. 

## Key objectives:
1.  Simulate a Realistic Diagnostic Process:  The interpretation should reflect how a doctor would analyze an ECG, ask clarifying questions, and arrive at a diagnosis. 
2.  Grounded ECG Understanding:  The analysis should be based on specific ECG features and explicitly reference these features as evidence. 
3.  Evidence-Based Reasoning:  The diagnosis should be supported by clear, logical reasoning tied to the ECG findings. 

## Guidelines for the ECG analysis: 
1.  Data: ECG embeddings generated by PCLR
2.  Act as a cardiologist and use medical knowledge to analyze the provided ECG data step-by-step: 

Initial Analysis:  Analyze the provided ECG image to identify key features such as rhythm, intervals, and any apparent abnormalities. 
Detailed Reasoni

In [ ]:
# Train DPO
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_out = trainer.train()
print(train_out)

# Generate a new report with the updated LoRA model
new_text = generate_report(trainer.model, tokenizer, x_prompt, max_new_tokens=450, temperature=0.7, top_p=0.95)

# Show after training
show_reports(
    prompt=x_prompt,
    baseline=baseline_text,
    gen1=gen1_text,
    gen2=gen2_text,
    chosen=chosen_text,
    rejected=rejected_text,
    ground_truth=gt_text,
    new_report=new_text,
    gen1_name=f"GEN 1 ({gen1_path.name})",
    gen2_name=f"GEN 2 ({gen2_path.name})",
    chosen_name=f"CHOSEN (Judge picked {winner}, {chosen_path.name})",
    rejected_name=f"REJECTED ({rejected_path.name})",
)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
1,0.693147
2,0.693147
3,0.693147


TrainOutput(global_step=3, training_loss=0.6931471824645996, metrics={'train_runtime': 3.185, 'train_samples_per_second': 0.942, 'train_steps_per_second': 0.942, 'total_flos': 0.0, 'train_loss': 0.6931471824645996, 'epoch': 3.0})
PROMPT

A.1 Diagnosis Guider Prompt 
# Your task:  Interpret the provided ECG data, identify key features and abnormalities in each lead, and generate a clinical diagnosis that is supported by the observed evidence. 

## Key objectives:
1.  Simulate a Realistic Diagnostic Process:  The interpretation should reflect how a doctor would analyze an ECG, ask clarifying questions, and arrive at a diagnosis. 
2.  Grounded ECG Understanding:  The analysis should be based on specific ECG features and explicitly reference these features as evidence. 
3.  Evidence-Based Reasoning:  The diagnosis should be supported by clear, logical reasoning tied to the ECG findings. 

## Guidelines for the ECG analysis: 
1.  Data: ECG embeddings generated by PCLR
2.  Act as a cardiolog

## Saving the finetuned model

In [ ]:
# After trainer.train()

save_dir = "mistral_cmr_dpo_lora"

trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"LoRA adapters saved to: {save_dir}")


LoRA adapters saved to: mistral_cmr_dpo_lora
